# 01e — Merge All External Data (Batch-by-Batch Flow)
**Purpose:** Combine extracted external data into enriched datasets, organized gradually by experiment batches.

We split the merge flow into distinct stages corresponding to our experiments:
1. **Batch 2 (External APIs):** Base + Elevation (01a) + SoilGrids (01b) + Weather (01c) + OSM (01d) -> Saves `*_enriched_exp2.parquet`
2. **Batch 3 & 5 (Spatial & Satellite++):** Batch 2 + HydroATLAS (01f) + RiverATLAS (01g) + SANLC (01h) + WorldPop (01i) + Sentinel-2 (01j) -> Saves `*_enriched.parquet` (Full)

**Inputs:** (add each notebook output as Kaggle dataset input)
- `train_base.parquet`, `val_base.parquet` — from notebook 00
- `elevation.parquet`, `soilgrids.parquet`, `weather.parquet`, `osm.parquet`
- `hydroatlas.parquet`, `riveratlas.parquet`, `sanlc.parquet`, `worldpop.parquet`, `sentinel.parquet`

**Outputs:**
- `train_enriched_exp2.parquet`, `val_enriched_exp2.parquet`
- `train_enriched.parquet`, `val_enriched.parquet` (Full)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import os, logging
import warnings
warnings.filterwarnings('ignore')

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-5s | %(message)s',
    datefmt='%H:%M:%S'
)
log = logging.getLogger('01e_merge')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'axes.titleweight': 'bold', 'font.size': 11})

OUTPUT_DIR = '/kaggle/working'

# Column config
LAT_COL     = 'Latitude'
LON_COL     = 'Longitude'
STATION_COL = 'station_id'
DATE_COL    = 'Sample Date'
TARGET_COLS = ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']

# Helper function to find input file path in Kaggle
def find_file(filename, default_dir='/kaggle/working'):
    target = os.path.join(default_dir, filename)
    if os.path.exists(target):
        return target
    input_dir = '/kaggle/input'
    if os.path.exists(input_dir):
        for root, _, files in os.walk(input_dir):
            if filename in files:
                return os.path.join(root, filename)
    raise FileNotFoundError(f"File {filename} not found in input/working directories.")

---
## 1. Load Base & External Parquet Files

In [ ]:
# Load base data
train_base = pd.read_parquet(find_file('train_base.parquet'))
val_base   = pd.read_parquet(find_file('val_base.parquet'))
log.info(f'Loaded Train base: {train_base.shape} | Val base: {val_base.shape}')

# Dictionary to hold loaded external datasets
external = {}
all_ext_sources = ['elevation', 'soilgrids', 'weather', 'osm', 'hydroatlas', 'riveratlas', 'sanlc', 'worldpop', 'sentinel']

for name in all_ext_sources:
    try:
        path = find_file(f'{name}.parquet')
        external[name] = pd.read_parquet(path)
        log.info(f'  Loaded {name:10s}: {external[name].shape}')
    except FileNotFoundError:
        log.warning(f'  Skipping {name:10s} (not found in inputs — placeholder values will be used if needed)')

---
## 2. Merge Stage 1 — Batch 2: External APIs (Exp 2)
Merges Elevation, SoilGrids, Weather, and OSM.

In [ ]:
station_keys = [STATION_COL, LAT_COL, LON_COL]
sample_keys  = [STATION_COL, LAT_COL, LON_COL, DATE_COL]
feature_counts = {}  # key: source, value: features added

train_exp2 = train_base.copy()
val_exp2   = val_base.copy()

log.info('--- MERGING BATCH 2 (EXTERNAL APIs) ---')

# Merge static APIs
for name in ['elevation', 'soilgrids', 'osm']:
    if name not in external:
        log.warning(f'  Skipping {name} (not found)')
        continue
    
    df = external[name]
    new_cols = [c for c in df.columns if c not in station_keys]
    if new_cols:
        n_before = train_exp2.shape[1]
        train_exp2 = train_exp2.merge(df[station_keys + new_cols], on=station_keys, how='left')
        val_exp2   = val_exp2.merge(df[station_keys + new_cols], on=station_keys, how='left')
        n_added = train_exp2.shape[1] - n_before
        feature_counts[name.title()] = n_added
        log.info(f'  + {name.title():10s}: {n_added} features merged')

# Merge temporal API (Weather)
if 'weather' in external:
    df = external['weather']
    new_cols = [c for c in df.columns if c not in sample_keys]
    if new_cols:
        n_before = train_exp2.shape[1]
        train_exp2 = train_exp2.merge(df[sample_keys + new_cols], on=sample_keys, how='left')
        val_exp2   = val_exp2.merge(df[sample_keys + new_cols], on=sample_keys, how='left')
        n_added = train_exp2.shape[1] - n_before
        feature_counts['Weather'] = n_added
        log.info(f'  + Weather   : {n_added} features merged')

# Save Exp 2 outputs
train_exp2_path = f'{OUTPUT_DIR}/train_enriched_exp2.parquet'
val_exp2_path   = f'{OUTPUT_DIR}/val_enriched_exp2.parquet'
train_exp2.to_parquet(train_exp2_path, index=False)
val_exp2.to_parquet(val_exp2_path, index=False)

log.info(f'Saved Exp 2 Train: {train_exp2.shape} | Val: {val_exp2.shape}')

---
## 3. Merge Stage 2 — Batch 3 & 5: Spatial Context & Sentinel-2 (Exp 3 & 5)
Merges HydroATLAS, RiverATLAS, SANLC, WorldPop, and Sentinel-2 onto the Batch 2 data.

In [ ]:
train_exp3 = train_exp2.copy()
val_exp3   = val_exp2.copy()

log.info('--- MERGING BATCH 3 & 5 (SPATIAL & SATELLITE++) ---')

for name in ['hydroatlas', 'riveratlas', 'sanlc', 'worldpop', 'sentinel']:
    if name not in external:
        log.warning(f'  Skipping {name} (not found)')
        continue
    
    df = external[name]
    new_cols = [c for c in df.columns if c not in station_keys]
    if new_cols:
        n_before = train_exp3.shape[1]
        train_exp3 = train_exp3.merge(df[station_keys + new_cols], on=station_keys, how='left')
        val_exp3   = val_exp3.merge(df[station_keys + new_cols], on=station_keys, how='left')
        n_added = train_exp3.shape[1] - n_before
        feature_counts[name.title()] = n_added
        log.info(f'  + {name.title():10s}: {n_added} features merged')

# Save final outputs (standard enriched parquet for downstream EDA/modeling)
train_final_path = f'{OUTPUT_DIR}/train_enriched.parquet'
val_final_path   = f'{OUTPUT_DIR}/val_enriched.parquet'
train_exp3.to_parquet(train_final_path, index=False)
val_exp3.to_parquet(val_final_path, index=False)

log.info(f'Saved Final Train: {train_exp3.shape} | Val: {val_exp3.shape}')

---
## Visualizations & Verification

In [ ]:
# FIGURE 1: Features per Data Source
if feature_counts:
    fig, ax = plt.subplots(figsize=(12, 5))
    names = list(feature_counts.keys())
    counts = list(feature_counts.values())
    palette = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336', '#00BCD4', '#E91E63', '#607D8B', '#795548']
    
    bars = ax.bar(names, counts, color=palette[:len(names)], edgecolor='white', linewidth=1.5)
    ax.bar_label(bars, fontsize=12, fontweight='bold')
    
    ax.set_ylabel('Number of Features')
    ax.set_title(f'Features per External Data Source (total = {sum(counts)})')
    ax.grid(axis='y', alpha=0.3)
    plt.xticks(rotation=25, ha='right')
    
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/fig_01e_features_per_source.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# FIGURE 2: Data Coverage Map
check_cols = {
    'Elevation': 'elevation_m',
    'SoilGrids': 'soil_phh2o',
    'OSM':       'osm_total_5000m',
    'HydroATLAS': 'basin_upstream_area_km2',
    'RiverATLAS': 'river_avg_discharge_cms',
    'SANLC':     'sanlc2022_class_1km',
    'WorldPop':  'worldpop_mean_1km',
    'Sentinel-2': 'sentinel_ndwi'
}

valid_checks = {k: v for k, v in check_cols.items() if v in train_exp3.columns}
n_panels = len(valid_checks)

if n_panels > 0:
    fig, axes = plt.subplots(2, int(np.ceil(n_panels/2)), figsize=(18, 12))
    axes = axes.flatten()
    
    for i, (src, col) in enumerate(valid_checks.items()):
        ax = axes[i]
        station_vals = train_exp3.groupby(STATION_COL)[col].first()
        station_locs = train_exp3.groupby(STATION_COL)[[LAT_COL, LON_COL]].first()
        has_data = station_vals.notna() & (station_vals != 'Unclassified/No Data')
        
        colors = ['#4CAF50' if v else '#F44336' for v in has_data]
        ax.scatter(station_locs[LON_COL], station_locs[LAT_COL],
                  c=colors, s=50, alpha=0.85, edgecolors='gray', linewidths=0.3)
        
        n_ok = has_data.sum()
        ax.set_title(f'{src}\n{n_ok}/{len(has_data)} stations', fontsize=12)
        ax.set_xlim(16, 33); ax.set_ylim(-35, -22)
        ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
        ax.grid(True, alpha=0.2)
    
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)
        
    legend_el = [mpatches.Patch(facecolor='#4CAF50', label='Has data'),
                 mpatches.Patch(facecolor='#F44336', label='Missing / Placeholder')]
    fig.legend(handles=legend_el, loc='lower center', ncol=2, fontsize=11, bbox_to_anchor=(0.5, -0.02))
    
    fig.suptitle('Data Coverage per Source', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/fig_01e_coverage_map.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Extraction Summary Table
summary_rows = []
for source, col in {**check_cols, 'Weather': 'precip_sum_7d'}.items():
    if col in train_exp3.columns:
        n_ok = train_exp3[col].notna().sum()
        n_total = len(train_exp3)
        summary_rows.append({
            'Source': source,
            'Features': feature_counts.get(source, 0) if source != 'Weather' else feature_counts.get('Weather', 0),
            'Coverage': f'{n_ok}/{n_total}',
            'Pct': f'{n_ok/n_total*100:.0f}%',
            'Status': 'OK' if n_ok == n_total else 'Partial'
        })
    else:
        summary_rows.append({
            'Source': source, 'Features': 0, 'Coverage': '—', 'Pct': '—', 'Status': 'NOT EXTRACTED'
        })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

In [ ]:
print('\n=== BATCH MERGE PROCESS COMPLETE ===')
print(f'Stage 1 (Exp 2) Output: train_enriched_exp2.parquet (Features: {train_exp2.shape[1] - train_base.shape[1]})')
print(f'Stage 2 (Exp 3 & 5) Output: train_enriched.parquet (Features: {train_exp3.shape[1] - train_base.shape[1]})')
print(f'Final Shape Train: {train_exp3.shape} | Val: {val_exp3.shape}')